# 03 - Feature Engineering, Leakage Detection & Splitting
In this comprehensive notebook, we engineer EVERY feature required by the project specifications, rigorously check for data leakage, and chronologically split our data.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

cleaned_data_path = '../../../data/processed/attendance_cleaned.csv'
df = pd.read_csv(cleaned_data_path)

# Ensure Date is properly formatted
df['Date'] = pd.to_datetime(df['Date'])
df['Start_Time_DT'] = pd.to_datetime(df['Start_Time'], errors='coerce').dt.time

# ⚠️ CRITICAL: Must sort chronologically before any feature engineering to prevent leakage!
df.sort_values(by=['Date', 'Start_Time_DT'], inplace=True)
df.reset_index(drop=True, inplace=True)
df.head()

,Date,Day_of_Week,Lecture_Number,Start_Time,End_Time,Subject,Faculty_ID,Semester,Section,Classroom,Total_Enrolled,Students_Present,Practical_Theory,Internal_Test_Week,Assignment_Due,Holiday_Before_After,Weather,Start_Time_DT,Attendance_Percentage
0,2026-06-15,Monday,1,10:00:00,11:00:00,Innovation & Entrepreneurship Development,F008,0,A&B,Computer Lab,204,143,Theory,No,No,No,Rainy,10:00:00,70.10
1,2026-06-15,Monday,2,10:15:00,11:15:00,Data Science & Machine Learning,F003,0,A&B,Computer Lab,204,150,Theory,No,No,No,Rainy,10:15:00,73.53
2,2026-06-15,Monday,3,10:30:00,11:30:00,Data Science & Machine Learning,F003,0,A&B,Computer Lab,204,156,Theory,No,No,No,Cloudy,10:30:00,76.47
3,2026-06-15,Monday,4,11:15:00,12:15:00,Software Testing & Quality Assurance,F009,0,A&B,Computer Lab,204,165,Theory,No,No,No,Sunny,11:15:00,80.88
4,2026-06-15,Monday,5,13:30:00,14:30:00,DS & ML Practical,F006,0,A&B,Computer Lab,204,181,Practical,No,No,No,Sunny,13:30:00,88.73


### Step 1: Global Temporal Features
Features related to the passage of time across the entire semester.

In [2]:
# 1. Day of Semester (0 on the first recorded day, increasing sequentially)
semester_start = df['Date'].min()
df['Day_of_Semester'] = (df['Date'] - semester_start).dt.days

# 2. Week Number (Standard ISO Calendar Week)
df['Week_Number'] = df['Date'].dt.isocalendar().week.astype(int)

# 3. Time-of-Day Clusters (Morning vs Afternoon)
df['Time_of_Day_Cluster'] = df['Start_Time_DT'].apply(lambda x: 'Morning' if x.hour < 12 else 'Afternoon')

# 3.5 Lunch Break Flag (Classes starting between 1:00 PM and 2:59 PM)
df['Is_Post_Lunch_Class'] = df['Start_Time_DT'].apply(lambda x: 1 if 13 <= x.hour <= 14 else 0)

df[['Date', 'Start_Time', 'Day_of_Semester', 'Week_Number', 'Time_of_Day_Cluster', 'Is_Post_Lunch_Class']].head()

,Date,Start_Time,Day_of_Semester,Week_Number,Time_of_Day_Cluster,Is_Post_Lunch_Class
0,2026-06-15,10:00:00,0,25,Morning,0
1,2026-06-15,10:15:00,0,25,Morning,0
2,2026-06-15,10:30:00,0,25,Morning,0
3,2026-06-15,11:15:00,0,25,Morning,0
4,2026-06-15,13:30:00,0,25,Afternoon,1


### Step 2: Subject-Specific Historical Tracking (Using `.shift()`)
We calculate historical features *per subject and section* using only past data.

In [3]:
group = df.groupby(['Subject', 'Section'])
global_mean_attendance = df['Attendance_Percentage'].mean()

# 4. Previous Lecture Attendance (Same Subject Previous Attendance)
df['Previous_Lecture_Attendance_Pct'] = group['Attendance_Percentage'].shift(1)
df['Previous_Lecture_Attendance_Pct'].fillna(0, inplace=True)

# 4.5 Last Session Attendance (Chronological Momentum by any faculty)
df['Last_Session_Attendance_Pct'] = df['Attendance_Percentage'].shift(1)
df['Last_Session_Attendance_Pct'].fillna(0, inplace=True)

# 5. Days Since Previous Lecture (Gap)
df['Prev_Date'] = group['Date'].shift(1)
df['Gap_Since_Previous_Lecture_Days'] = (df['Date'] - df['Prev_Date']).dt.days
df['Gap_Since_Previous_Lecture_Days'] = df['Gap_Since_Previous_Lecture_Days'].fillna(0).astype(int)
df.drop(columns=['Prev_Date'], inplace=True)

# 6. Rolling Average (Last 3 Lectures)
df['Rolling_Avg_3_Lectures'] = group['Attendance_Percentage'].transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
df['Rolling_Avg_3_Lectures'].fillna(0, inplace=True)

# 7. Consecutive Lecture Count (Streak of classes with <= 2 days gap)
df['Is_Consecutive'] = (df['Gap_Since_Previous_Lecture_Days'] <= 2).astype(int)
df['Consecutive_Lecture_Count'] = group['Is_Consecutive'].cumsum()

# 8. Monthly Expanding Average
df['Month'] = df['Date'].dt.month
df['Monthly_Expanding_Mean'] = df.groupby(['Month', 'Subject', 'Section'])['Attendance_Percentage'].transform(lambda x: x.shift(1).expanding().mean())
df['Monthly_Expanding_Mean'].fillna(0, inplace=True)

0       0.000000
1       0.000000
2      73.530000
3       0.000000
4       0.000000
         ...    
510    71.242778
511    71.255000
512    71.120833
513    79.535000
514    77.025333
Name: Monthly_Expanding_Mean, Length: 515, dtype: float64

### Step 3: Event Flags & Categorical Encoding
Converting textual disruptions into machine-readable numeric flags, and one-hot encoding categories.

In [4]:
# 9. Holiday Adjacent Flag
df['Is_Holiday_Adjacent'] = df['Holiday_Before_After'].apply(lambda x: 1 if str(x).lower() == 'yes' else 0)

# 10. Week-Before-Exam Flag
df['Week_Before_Exam_Flag'] = df['Internal_Test_Week'].apply(lambda x: 1 if str(x).lower() == 'yes' else 0)

# 11. One-Hot Encoding for remaining categorical string variables
cols_to_encode = ['Day_of_Week', 'Subject', 'Weather', 'Practical_Theory', 'Time_of_Day_Cluster']
df_encoded = pd.get_dummies(df, columns=cols_to_encode, drop_first=True)

print(f"Final Engineered Dataset Shape: {df_encoded.shape}")

Final Engineered Dataset Shape: (515, 59)


### Step 4: Leakage Detection Gate
Mathematically proving that our engineered historical features do not accidentally contain the future `Attendance_Percentage`.

In [5]:
engineered_numeric_cols = [
    'Previous_Lecture_Attendance_Pct', 'Gap_Since_Previous_Lecture_Days', 
    'Rolling_Avg_3_Lectures', 'Consecutive_Lecture_Count', 'Monthly_Expanding_Mean',
    'Day_of_Semester', 'Is_Post_Lunch_Class'
]

correlation_matrix = df_encoded[['Attendance_Percentage'] + engineered_numeric_cols].corr()
print("Correlation with Target (Attendance_Percentage):")
print(correlation_matrix['Attendance_Percentage'])

for col in engineered_numeric_cols:
    corr = correlation_matrix.loc[col, 'Attendance_Percentage']
    assert corr < 0.95, f"LEAKAGE DETECTED! {col} is perfectly correlated with the target."

print("\n✅ LEAKAGE DETECTION PASSED: No engineered feature is illegally predicting the target.")

Correlation with Target (Attendance_Percentage):
Attendance_Percentage              1.000000
Previous_Lecture_Attendance_Pct    0.464144
Gap_Since_Previous_Lecture_Days   -0.124751
Rolling_Avg_3_Lectures             0.519285
Consecutive_Lecture_Count         -0.003176
Monthly_Expanding_Mean             0.514852
Day_of_Semester                   -0.044577
Is_Post_Lunch_Class                0.057834
Name: Attendance_Percentage, dtype: float64

✅ LEAKAGE DETECTION PASSED: No engineered feature is illegally predicting the target.


### Step 5: Temporal Splitting (Chronological)
Splitting into Train (70%), Validation (15%), and Test (15%) strictly based on time.

In [6]:
from sklearn.model_selection import train_test_split

drop_cols = ['Start_Time', 'End_Time', 'Faculty_ID', 'Branch', 'Section', 'Classroom', 'Holiday_Before_After', 'Internal_Test_Week', 'Special_Event', 'Start_Time_DT', 'Previous_Lecture_Attendance', 'Gap_Since_Previous_Lecture', 'Assignment_Due', 'Attendance_Category', 'Lecture_No']
df_final = df_encoded.drop(columns=[col for col in drop_cols if col in df_encoded.columns])

# 70% Train, 30% Temp (Val + Test)
train_df, temp_df = train_test_split(df_final, test_size=0.30, shuffle=False)

# 15% Val, 15% Test
val_df, test_df = train_test_split(temp_df, test_size=0.50, shuffle=False)

print(f"Train set: {train_df.shape}")
print(f"Validation set: {val_df.shape}")
print(f"Test set: {test_df.shape}")

assert train_df['Date'].max() <= val_df['Date'].min(), "Temporal Leakage: Train data overlaps Validation data!"
assert val_df['Date'].max() <= test_df['Date'].min(), "Temporal Leakage: Validation data overlaps Test data!"
print("\n✅ TEMPORAL SPLITS VERIFIED: Data is strictly chronological. No future peeking.")

train_df = train_df.drop(columns=['Date'])
val_df = val_df.drop(columns=['Date'])
test_df = test_df.drop(columns=['Date'])

train_df.to_csv('../../../data/processed/train.csv', index=False)
val_df.to_csv('../../../data/processed/val.csv', index=False)
test_df.to_csv('../../../data/processed/test.csv', index=False)

Train set: (360, 50)
Validation set: (77, 50)
Test set: (78, 50)

✅ TEMPORAL SPLITS VERIFIED: Data is strictly chronological. No future peeking.
